<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/split_wise3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import heapq

def settle(balances, optimize=False):
    debts, creds, txns = [], [], []
    for i, b in enumerate(balances):
        if b > 0: creds.append((b,i))
        elif b < 0: debts.append((b,i))
    subsets = [(creds, debts)]
    if optimize: subsets = max_0_subsets(tuple(subsets[0][0]), tuple(subsets[0][1]))
    for subset in subsets:
        txns += heap_settler(*subset)
    return txns

def heap_settler(creds, debts):
    txns = []
    creds = [(-x[0], x[1]) for x in creds]
    heapq.heapify(creds)
    heapq.heapify(debts)
    while creds or debts:
        cb,ci = heapq.heappop(creds)
        db,di = heapq.heappop(debts)
        bal = -max(cb,db)
        txns.append((di, ci, bal))
        if cb > db: heapq.heappush(debts, (db+bal, di))
        elif cb < db: heapq.heappush(creds, (cb+bal, ci))
    return txns


In [22]:
print(settle([8,-8,7,3,-10]))
print(settle([-8,5,3,-7,7]))
print(settle([+12, +11, +10, +9, -15, -14, -8, -5]))

[(4, 0, 8), (1, 2, 7), (4, 3, 2), (1, 3, 1)]
[(0, 4, 7), (3, 1, 5), (3, 2, 2), (0, 2, 1)]
[(4, 0, 12), (5, 1, 11), (6, 2, 8), (7, 3, 5), (4, 3, 3), (5, 2, 2), (5, 3, 1)]


In [23]:
from itertools import product
from functools import cache

def powset(s):
    for i in range(1, 2**len(s)):
        x1 = [s[j] for j in range(len(s)) if i & 2**j]
        x2 = [s[j] for j in range(len(s)) if not (i & 2**j)]
        yield x1, x2

@cache
def max_0_subsets(creds, debts):
    subsets = []
    if not creds or not debts: return subsets
    for cset, dset in product(powset(creds), powset(debts)):
        s = sum(x[0] for x in cset[0]) + sum(x[0] for x in dset[0])
        curr = []
        if s == 0:
            curr = [(cset[0], dset[0])]
            if len(cset[1]) == 1 or len(dset[1]) == 1:
                curr += [(cset[1], dset[1])]
            else:
                curr += max_0_subsets(tuple(cset[1]), tuple(dset[1]))
        if len(curr) > len(subsets):
            subsets = curr
    return subsets

In [24]:
#list(powset("ABCD"))
print(settle([+12, +11, +10, +9, -15, -14, -8, -5], True))
max_0_subsets.cache_info()

[(4, 0, 12), (6, 1, 8), (4, 1, 3), (5, 2, 10), (7, 3, 5), (5, 3, 4)]


CacheInfo(hits=4, misses=6, maxsize=None, currsize=6)